<a href="https://colab.research.google.com/github/nayankhanal/ml-class/blob/vs-code/pytorch/rnn_based_qa_system_class_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [185]:
import pandas as pd

In [186]:
df = pd.read_csv("100_Unique_QA_Dataset.csv")

In [187]:
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [188]:
df['question'], df['answer'] = df['question'].str.lower(), df['answer'].str.lower()

In [189]:
import re

In [190]:
df[['question', 'answer']] = df[['question', 'answer']].apply(lambda col: col.str.replace(r'<.*?>', '', regex=True))

In [191]:
url_pattern = r'http[s]?://\S+|www\.\S+'

df['question'] = df['question'].str.replace(url_pattern, '', regex=True)
df['answer'] = df['answer'].str.replace(url_pattern, '', regex=True)

In [192]:
import string

punct_pattern = f'[{string.punctuation}]'

df['question'] = df['question'].str.replace(punct_pattern, ' ', regex=True)
# df['answer'] = df['answer'].str.replace(punct_pattern, ' ', regex=True)

In [193]:
# pip install pyspellchecker

In [194]:
from spellchecker import SpellChecker

spell = SpellChecker()

misspelled_words = set()

def find_misspelled(text):
  words = text.split()
  unknown = spell.unknown(words)
  misspelled_words.update(unknown)

In [195]:
df['question'].apply(find_misspelled)
# df['answer'].apply(find_misspelled)

,question
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [196]:
misspelled_words

{'ii', 'tv', 'uk'}

In [197]:
# pip install emoji

In [198]:
import emoji

emoji_list = []

def find_emojis(text):
    found = emoji.emoji_list(text)
    for item in found:
        emoji_list.append(item['emoji'])

df['question'].apply(find_emojis)
df['answer'].apply(find_emojis)

print(emoji_list)

[]


In [199]:
# import emoji

# def handle_emojis(text):
#     return emoji.demojize(text)

# df['question'] = df['question'].apply(handle_emojis)
# df['answer'] = df['answer'].apply(handle_emojis)

In [200]:
df.head()

,question,answer
0,what is the capital of france,paris
1,what is the capital of germany,berlin
2,who wrote to kill a mockingbird,harper-lee
3,what is the largest planet in our solar system,jupiter
4,what is the boiling point of water in celsius,100


In [201]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [202]:
nltk.word_tokenize(df['question'][0])

['what', 'is', 'the', 'capital', 'of', 'france']

In [203]:
# vocab
vocab = {'<UNK>':0}

In [204]:
def build_vocab(row):
  tokenized_question = nltk.word_tokenize(row['question'])
  tokenized_answer = nltk.word_tokenize(row['answer'])

  merged_tokens = tokenized_question + tokenized_answer

  for token in merged_tokens:
    if token not in vocab:
      vocab[token] = len(vocab)

In [205]:
df.apply(build_vocab, axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [206]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'mockingbird': 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 '1984': 67,
 'george-orwell': 68,
 'currency': 69,
 'unit

In [207]:
len(vocab)

324

In [242]:
# convert words to numerical indices
def text_to_indices(text, vocab):
  indexed_text = []

  text = text.lower()
  text = re.sub(punct_pattern, ' ', text)

  for token in nltk.word_tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text

In [209]:
text_to_indices("What is campusx", vocab)

[1, 2, 0]

In [210]:
import torch
from torch.utils.data import Dataset, DataLoader

In [211]:
class QADataset(Dataset):
  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):
    numerical_questions = text_to_indices(self.df.iloc[index]['question'], self.vocab)
    numerical_answers = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

    return torch.tensor(numerical_questions), torch.tensor(numerical_answers)

In [212]:
dataset = QADataset(df, vocab)

In [213]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [214]:
for question, answer in dataloader:
  print(question, answer[0])

tensor([[10, 55,  3, 56,  5, 57]]) tensor([58])
tensor([[  1,   2,   3, 103,   5, 104,  19, 105]]) tensor([106])
tensor([[  1,   2,   3,  37, 133,   5,  26]]) tensor([134])
tensor([[1, 2, 3, 4, 5, 6]]) tensor([7])
tensor([[ 42, 263, 264,  14, 265, 266, 158, 267]]) tensor([268])
tensor([[  1,   2,   3,  69,   5, 155]]) tensor([156])
tensor([[  1,   2,   3, 221,   5, 222, 223, 224]]) tensor([225])
tensor([[  1,   2,   3,   4,   5, 279]]) tensor([280])
tensor([[ 1,  2,  3,  4,  5, 99]]) tensor([100])
tensor([[42, 86, 87, 88, 89, 39, 90]]) tensor([91])
tensor([[ 42, 174,   2,  62,  39, 175, 176,  12, 177, 178]]) tensor([179])
tensor([[ 42, 137,   2, 226,  12,   3, 227, 228]]) tensor([155])
tensor([[10, 29,  3, 30, 31]]) tensor([32])
tensor([[ 1,  2,  3, 17, 18, 19, 20, 21, 22]]) tensor([23])
tensor([[ 10,  11, 157, 158, 159]]) tensor([160])
tensor([[ 42, 107,   2, 108,  19, 109]]) tensor([110])
tensor([[ 42,   2,   3, 274, 211, 275]]) tensor([276])
tensor([[ 42, 250, 251, 118, 252, 253]]) 

In [215]:
import torch.nn as nn

In [216]:
class SimpleRNN(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, 50)
    self.rnn = nn.RNN(50, 64, batch_first=True)
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, question):
    embedded_question = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))

    return output

In [217]:
dataset[0][0]

tensor([1, 2, 3, 4, 5, 6])

In [218]:
embed = nn.Embedding(324, embedding_dim=50)

In [219]:
a = embed(dataset[0][0])

In [220]:
a.shape

torch.Size([6, 50])

In [221]:
rnn = nn.RNN(50, 64)

In [222]:
b = rnn(a)

In [223]:
b[0].shape

torch.Size([6, 64])

In [224]:
b[1].shape

torch.Size([1, 64])

In [225]:
fc = nn.Linear(64, 324)

In [226]:
c = fc(b[1])

In [227]:
c.shape

torch.Size([1, 324])

In [228]:
learning_rate = 0.001
epochs = 20

In [229]:
model = SimpleRNN(len(vocab))

In [230]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [231]:
# training loop
for epoch in range(epochs):

  total_loss = 0

  for question, answer in dataloader:

    optimizer.zero_grad()

    # forward pass
    output = model(question)

    # loss -> output shape (1,324) - (1)
    loss = criterion(output, answer[0])

    # gradients
    loss.backward()

    # update parameters
    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch+1}, Avg Loss: {total_loss/len(dataloader)}")

Epoch: 1, Avg Loss: 5.820882272720337
Epoch: 2, Avg Loss: 5.115199512905544
Epoch: 3, Avg Loss: 4.294067396057977
Epoch: 4, Avg Loss: 3.5090469148423935
Epoch: 5, Avg Loss: 2.898341663678487
Epoch: 6, Avg Loss: 2.3467068089379204
Epoch: 7, Avg Loss: 1.8519449048572116
Epoch: 8, Avg Loss: 1.4217510236634148
Epoch: 9, Avg Loss: 1.0742477314339745
Epoch: 10, Avg Loss: 0.8099907467762629
Epoch: 11, Avg Loss: 0.6181390325228373
Epoch: 12, Avg Loss: 0.4842589853538407
Epoch: 13, Avg Loss: 0.38196246557765534
Epoch: 14, Avg Loss: 0.3151709978779157
Epoch: 15, Avg Loss: 0.26218160217007
Epoch: 16, Avg Loss: 0.2214743105901612
Epoch: 17, Avg Loss: 0.1860801786184311
Epoch: 18, Avg Loss: 0.16299816154771382
Epoch: 19, Avg Loss: 0.14291110634803772
Epoch: 20, Avg Loss: 0.12527864111794365


In [232]:
def predict(model, question, threshold=0.5):

  # convert question to numbers
  numerical_question = text_to_indices(question, vocab)

  # tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output = model(question_tensor)

  # convert logits to probs
  probs = torch.nn.functional.softmax(output, dim=1)

  # find index of max prob
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("I don't know")

  print(list(vocab.keys())[index])

In [243]:
predict(model, "What is the largest planet in our solar system?")

jupiter
